## 1. Imports and Setup


In [ ]:
pip install lpips


In [ ]:

import os
import copy
import random
import glob
import shutil
import time
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.nn.utils import spectral_norm
from torch import amp

import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.models import vgg19, VGG19_Weights

import lpips


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Configuration


In [ ]:


class Config:

    pretrained_model_path  = ''
    resume_checkpoint_path = None  

    train_hr_dir = ''
    train_lr_dir = ''
    val_hr_dir   = ''
    val_lr_dir   = ''

    checkpoint_dir = 'experiments/checkpoints'
    log_dir        = 'experiments/logs'
    prev_log_dir   = None   # Point to previous session's log_dir on day 2+ for TB continuity

    # Generator architecture 
    num_in_ch    = 3
    num_out_ch   = 3
    scale_factor = 4
    num_feat     = 64
    num_block    = 23
    num_grow_ch  = 32

   # Channel Attention
    ca_reduction       = 8
    use_ca_after_trunk = True
    use_ca_after_up1   = True
    use_ca_after_up2   = True

    #Discriminator 
    disc_num_feat = 64
    discriminator_type = "patchgan"  # "patchgan" or "unet"
    freeze_discriminator_always          = False
    freeze_discriminator_start_epoch     = None
    freeze_discriminator_end_epoch       = None

    # Training schedule
    num_epochs    = 18
    hr_patch_size = 192
    lr_patch_size = 48      # = 192 // 4

   
    batch_size       = 8
    grad_accum_steps = 2

    num_workers        = 4
    persistent_workers = True
    prefetch_factor    = 4

    # ── Phase schedule ───────────────────────────────────────────────────
    gan_start_epoch      = 10
    gan_rampup_end_epoch = 18  

    # Learning rates
    lr_g    = 5e-6
    lr_g_ca = 1e-4
    lr_d    = 1e-5
    betas   = (0.9, 0.99)

    #LR Scheduler 
    lr_min = 5e-8

    #Loss weights
    lambda_pixel      = 1.0
    lambda_perceptual = 0.8
    lambda_gan_low    = 0.005  
    lambda_gan_high   = 0.1    

    #  EMA 
    ema_decay = 0.9995

    # Checkpointing and logging
    save_interval      = 3
    val_interval       = 1
    log_interval       = 30
    image_log_interval = 3
    tb_flush_every_epoch = False


config = Config()

os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(config.log_dir, exist_ok=True)
print('Configuration loaded.')


In [ ]:

# Verify Paths
print(f"Pretrained model : {config.pretrained_model_path}")
print(f"  exists         : {os.path.exists(config.pretrained_model_path)}")
print(f"Train HR dir     : {config.train_hr_dir}")
print(f"  exists         : {os.path.exists(config.train_hr_dir)}")
print(f"Train LR dir     : {config.train_lr_dir}")
print(f"  exists         : {os.path.exists(config.train_lr_dir)}")
print(f"Val HR dir       : {config.val_hr_dir}")
print(f"  exists         : {os.path.exists(config.val_hr_dir)}")
print(f"Val LR dir       : {config.val_lr_dir}")
print(f"  exists         : {os.path.exists(config.val_lr_dir)}")
print(f"\nHR patch: {config.hr_patch_size}  |  LR patch: {config.lr_patch_size}  |  Scale: {config.scale_factor}x")
print(f"Batch size: {config.batch_size}  |  Accum steps: {config.grad_accum_steps}  |  Epochs: {config.num_epochs}")
print(f"CA reduction ratio: {config.ca_reduction}")
print(
    f"CA switches -> trunk: {config.use_ca_after_trunk}, "
    f"up1: {config.use_ca_after_up1}, up2: {config.use_ca_after_up2}"
)
print(f"GAN schedule -> start: {config.gan_start_epoch}, ramp end: {config.gan_rampup_end_epoch}, low: {config.lambda_gan_low}, high: {config.lambda_gan_high}")
print(
    f"Discriminator freeze -> always: {config.freeze_discriminator_always}, "
    f"range: [{config.freeze_discriminator_start_epoch}, {config.freeze_discriminator_end_epoch}]"
)


## 3. Dataset and DataLoaders


In [ ]:

class DRealSRPairedDataset(Dataset):
   

    def __init__(self, hr_dir, lr_dir, hr_patch_size=256, scale_factor=4, augment=True):
        self.hr_patch_size = hr_patch_size
        self.lr_patch_size = hr_patch_size // scale_factor
        self.scale_factor  = scale_factor
        self.augment       = augment
        self.to_tensor     = transforms.ToTensor()

        hr_paths = sorted(Path(hr_dir).glob('*'))
        hr_paths = [p for p in hr_paths if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')]

        self.pairs = []
        for hr_p in hr_paths:
            lr_name = hr_p.name.replace('x4', 'x1')
            lr_p = Path(lr_dir) / lr_name
            if lr_p.exists():
                self.pairs.append((hr_p, lr_p))

        print(f"Found {len(self.pairs)} paired images  (HR: {hr_dir}, LR: {lr_dir})")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        hr_path, lr_path = self.pairs[idx]
        hr_img = Image.open(hr_path).convert('RGB')
        lr_img = Image.open(lr_path).convert('RGB')

        if self.augment:
            hr_w, hr_h = hr_img.size
            max_x = max(0, hr_w - self.hr_patch_size)
            max_y = max(0, hr_h - self.hr_patch_size)
            x = random.randint(0, max_x)
            y = random.randint(0, max_y)

            hr_img = hr_img.crop((x, y, x + self.hr_patch_size, y + self.hr_patch_size))

            lx = x // self.scale_factor
            ly = y // self.scale_factor
            lr_img = lr_img.crop((lx, ly, lx + self.lr_patch_size, ly + self.lr_patch_size))

            if random.random() > 0.5:
                hr_img = TF.hflip(hr_img)
                lr_img = TF.hflip(lr_img)

            if random.random() > 0.5:
                hr_img = TF.vflip(hr_img)
                lr_img = TF.vflip(lr_img)

            if random.random() > 0.5:
                angle = random.choice([90, 180, 270])
                hr_img = TF.rotate(hr_img, angle)
                lr_img = TF.rotate(lr_img, angle)

           
            if random.random() > 0.5:
                cj = transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05, hue=0.02)
                fn_idx, brightness_factor, contrast_factor, saturation_factor, hue_factor = transforms.ColorJitter.get_params(
                    cj.brightness, cj.contrast, cj.saturation, cj.hue)

                def _apply_color_jitter(img):
                    for fn_id in fn_idx:
                        if fn_id == 0 and brightness_factor is not None:
                            img = TF.adjust_brightness(img, brightness_factor)
                        elif fn_id == 1 and contrast_factor is not None:
                            img = TF.adjust_contrast(img, contrast_factor)
                        elif fn_id == 2 and saturation_factor is not None:
                            img = TF.adjust_saturation(img, saturation_factor)
                        elif fn_id == 3 and hue_factor is not None:
                            img = TF.adjust_hue(img, hue_factor)
                    return img

                hr_img = _apply_color_jitter(hr_img)
                lr_img = _apply_color_jitter(lr_img)
        else:
            hr_w, hr_h = hr_img.size
            cx = max(0, (hr_w - self.hr_patch_size) // 2)
            cy = max(0, (hr_h - self.hr_patch_size) // 2)
            hr_img = hr_img.crop((cx, cy, cx + self.hr_patch_size, cy + self.hr_patch_size))
            lx = cx // self.scale_factor
            ly = cy // self.scale_factor
            lr_img = lr_img.crop((lx, ly, lx + self.lr_patch_size, ly + self.lr_patch_size))

        return self.to_tensor(lr_img), self.to_tensor(hr_img), hr_path.name


def create_dataloaders(cfg):
    train_ds = DRealSRPairedDataset(
        cfg.train_hr_dir, cfg.train_lr_dir,
        cfg.hr_patch_size, cfg.scale_factor, augment=True)
    val_ds = DRealSRPairedDataset(
        cfg.val_hr_dir, cfg.val_lr_dir,
        cfg.hr_patch_size, cfg.scale_factor, augment=False)

    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=True, drop_last=True,
        persistent_workers=cfg.persistent_workers,
        prefetch_factor=cfg.prefetch_factor)
    val_loader = DataLoader(
        val_ds, batch_size=1, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True,
        persistent_workers=cfg.persistent_workers,
        prefetch_factor=cfg.prefetch_factor)

    return train_loader, val_loader


## 4. Model Architecture with Channel Attention

### 4.1 Channel Attention Module (SE-style)




In [ ]:

class ChannelAttention(nn.Module):
  
    def __init__(self, num_feat: int, reduction: int = 8):
        super().__init__()
        mid = max(num_feat // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(num_feat, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, num_feat, bias=True),  
            nn.Sigmoid(),
        )
        
        nn.init.constant_(self.fc[2].weight, 0.0)
        nn.init.constant_(self.fc[2].bias,   3.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


### 4.2 Residual Dense Block with Channel Attention (RDB-CA)


In [ ]:

class ResidualDenseBlock(nn.Module):

    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat,                  num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat +   num_grow_ch,  num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2*num_grow_ch,  num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3*num_grow_ch,  num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4*num_grow_ch,  num_feat,    3, 1, 1)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
        self._init_weights()

    def _init_weights(self):
        for m in [self.conv1, self.conv2, self.conv3, self.conv4, self.conv5]:
            nn.init.kaiming_normal_(m.weight, a=0.2, mode='fan_in',
                                    nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        nn.init.constant_(self.conv5.weight, 0)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x


class ResidualDenseBlockCA(nn.Module):

    def __init__(self, num_feat=64, num_grow_ch=32, ca_reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat,                  num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat +   num_grow_ch,  num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2*num_grow_ch,  num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3*num_grow_ch,  num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4*num_grow_ch,  num_feat,    3, 1, 1)

        self.ca = ChannelAttention(num_feat, reduction=ca_reduction)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
        self._init_weights()

    def _init_weights(self):
        for m in [self.conv1, self.conv2, self.conv3, self.conv4, self.conv5]:
            nn.init.kaiming_normal_(m.weight, a=0.2, mode='fan_in',
                                    nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        nn.init.constant_(self.conv5.weight, 0)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        x5 = self.ca(x5)
        return x5 * 0.2 + x


### 4.3 RRDB with Channel Attention (RRDB-CA)



In [ ]:

class RRDB(nn.Module):

    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)

    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out * 0.2 + x


class RRDBCA(nn.Module):

    def __init__(self, num_feat=64, num_grow_ch=32, ca_reduction=16):
        super().__init__()
        self.rdb1 = ResidualDenseBlockCA(num_feat, num_grow_ch, ca_reduction)
        self.rdb2 = ResidualDenseBlockCA(num_feat, num_grow_ch, ca_reduction)
        self.rdb3 = ResidualDenseBlockCA(num_feat, num_grow_ch, ca_reduction)

        self.ca = ChannelAttention(num_feat, reduction=ca_reduction)

    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        out = self.ca(out)
        return out * 0.2 + x


### 4.4 RRDBNet-CA Generator

Architecture flow: RRDB trunk (23 blocks) → CA → Upsample 1 → CA → Upsample 2 → CA → Output


In [ ]:

class RRDBNetCA(nn.Module):

    def __init__(self, num_in_ch=3, num_out_ch=3, scale=4,
                 num_feat=64, num_block=23, num_grow_ch=32,
                 ca_reduction=16,
                 use_ca_after_trunk=True, use_ca_after_up1=True, use_ca_after_up2=True):
        super().__init__()
        self.scale = scale

        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)

    
        self.body = nn.Sequential(*[RRDB(num_feat=num_feat, num_grow_ch=num_grow_ch) for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)

        self.ca_trunk = ChannelAttention(num_feat, reduction=ca_reduction) if use_ca_after_trunk else nn.Identity()
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.ca_up1 = ChannelAttention(num_feat, reduction=ca_reduction) if use_ca_after_up1 else nn.Identity()
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.ca_up2 = ChannelAttention(num_feat, reduction=ca_reduction) if use_ca_after_up2 else nn.Identity()
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = self.conv_body(self.body(feat))
        feat = feat + body_feat

        feat = self.ca_trunk(feat)

        feat = F.interpolate(feat, scale_factor=2, mode='nearest')
        feat = self.lrelu(self.conv_up1(feat))
        feat = self.ca_up1(feat)

        feat = F.interpolate(feat, scale_factor=2, mode='nearest')
        feat = self.lrelu(self.conv_up2(feat))
        feat = self.ca_up2(feat)

        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out


In [ ]:

# Quick shape test
with torch.no_grad():
    _g = RRDBNetCA(num_in_ch=3, num_out_ch=3, scale=4,
                   num_feat=64, num_block=23, num_grow_ch=32,
                   ca_reduction=config.ca_reduction,
                   use_ca_after_trunk=config.use_ca_after_trunk,
                   use_ca_after_up1=config.use_ca_after_up1,
                   use_ca_after_up2=config.use_ca_after_up2)
    _x = torch.randn(1, 3, 64, 64)
    _y = _g(_x)
    total_params  = sum(p.numel() for p in _g.parameters())
    ca_params     = sum(p.numel() for n, p in _g.named_parameters() if '.ca_' in n or '.ca.' in n)
    print(f"Generator-CA  |  Input: {_x.shape}  ->  Output: {_y.shape}")
    print(f"Total parameters : {total_params:,}")
    print(f"CA-only params   : {ca_params:,}  ({100*ca_params/total_params:.2f}% of total)")
    print(
        f"CA switches -> trunk: {config.use_ca_after_trunk}, "
        f"up1: {config.use_ca_after_up1}, up2: {config.use_ca_after_up2}"
    )
    print(f"RRDB trunk blocks (standard): {config.num_block}")
    del _g, _x, _y


### 4.5 PatchGAN Discriminator


In [ ]:


class PatchGANDiscriminator(nn.Module):

    def __init__(self, num_in_ch=3, num_feat=64, n_layers=4):
        super().__init__()
        kw = 4
        padw = 1

        layers = [
            nn.Conv2d(num_in_ch, num_feat, kernel_size=kw, stride=2, padding=padw),
            nn.LeakyReLU(0.2, inplace=True),
        ]

        nf_mult = 1
        for n in range(1, n_layers):
            nf_mult_prev = nf_mult
            nf_mult = min(2 ** n, 8)
            layers += [
                spectral_norm(nn.Conv2d(num_feat * nf_mult_prev, num_feat * nf_mult, kernel_size=kw, stride=2, padding=padw, bias=False)),
                nn.LeakyReLU(0.2, inplace=True),
            ]

        nf_mult_prev = nf_mult
        nf_mult = min(2 ** n_layers, 8)
        layers += [
            spectral_norm(nn.Conv2d(num_feat * nf_mult_prev, num_feat * nf_mult, kernel_size=kw, stride=1, padding=padw, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(num_feat * nf_mult, 1, kernel_size=kw, stride=1, padding=padw, bias=False)),
        ]

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class UNetDiscriminatorSN(nn.Module):
    def __init__(self, num_in_ch=3, num_feat=64, skip_connection=True):
        super().__init__()
        self.skip_connection = skip_connection
        norm = spectral_norm
        self.conv0 = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)
        self.conv1 = norm(nn.Conv2d(num_feat, num_feat * 2, 4, 2, 1, bias=False))
        self.conv2 = norm(nn.Conv2d(num_feat * 2, num_feat * 4, 4, 2, 1, bias=False))
        self.conv3 = norm(nn.Conv2d(num_feat * 4, num_feat * 8, 4, 2, 1, bias=False))
        self.conv4 = norm(nn.Conv2d(num_feat * 8, num_feat * 4, 3, 1, 1, bias=False))
        self.conv5 = norm(nn.Conv2d(num_feat * 4, num_feat * 2, 3, 1, 1, bias=False))
        self.conv6 = norm(nn.Conv2d(num_feat * 2, num_feat, 3, 1, 1, bias=False))
        self.conv7 = norm(nn.Conv2d(num_feat, num_feat, 3, 1, 1, bias=False))
        self.conv8 = norm(nn.Conv2d(num_feat, num_feat, 3, 1, 1, bias=False))
        self.conv9 = nn.Conv2d(num_feat, 1, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x0 = self.lrelu(self.conv0(x))
        x1 = self.lrelu(self.conv1(x0))
        x2 = self.lrelu(self.conv2(x1))
        x3 = self.lrelu(self.conv3(x2))

        x3 = F.interpolate(x3, size=x2.shape[2:], mode="bilinear", align_corners=False)
        x4 = self.lrelu(self.conv4(x3))
        if self.skip_connection:
            x4 = x4 + x2

        x4 = F.interpolate(x4, size=x1.shape[2:], mode="bilinear", align_corners=False)
        x5 = self.lrelu(self.conv5(x4))
        if self.skip_connection:
            x5 = x5 + x1

        x5 = F.interpolate(x5, size=x0.shape[2:], mode="bilinear", align_corners=False)
        x6 = self.lrelu(self.conv6(x5))
        if self.skip_connection:
            x6 = x6 + x0

        out = self.lrelu(self.conv7(x6))
        out = self.lrelu(self.conv8(out))
        out = self.conv9(out)
        return out


def build_discriminator(cfg):
    disc_type = str(getattr(cfg, "discriminator_type", "patchgan")).lower()
    if disc_type == "patchgan":
        return PatchGANDiscriminator(num_in_ch=cfg.num_in_ch, num_feat=cfg.disc_num_feat)
    elif disc_type == "unet":
        return UNetDiscriminatorSN(num_in_ch=cfg.num_in_ch, num_feat=cfg.disc_num_feat)
    else:
        raise ValueError(f"Unsupported discriminator_type: {cfg.discriminator_type}. Use 'patchgan' or 'unet'.")


## 5. Load Pretrained Weights into CA Generator


In [ ]:

def load_pretrained_generator_ca(model, weight_path, device='cpu'):
   
    assert os.path.isfile(weight_path), f"Weights not found: {weight_path}"

    checkpoint = torch.load(weight_path, map_location=device, weights_only=False)

    if 'params_ema' in checkpoint:
        state_dict = checkpoint['params_ema']
        print('Loaded key: params_ema')
    elif 'params' in checkpoint:
        state_dict = checkpoint['params']
        print('Loaded key: params')
    else:
        state_dict = checkpoint
        print('Loaded raw state_dict')

    missing, unexpected = model.load_state_dict(state_dict, strict=False)

    ca_missing    = [k for k in missing if ('ca_' in k or '.ca.' in k)]
    other_missing = [k for k in missing if not ('ca_' in k or '.ca.' in k)]
    print(f'\nPretrained keys loaded  : {len(state_dict) - len(unexpected)}')
    print(f'New CA keys (init only) : {len(ca_missing)}')
    if other_missing:
        print(f'WARNING — unexpected missing keys (non-CA): {other_missing}')
    if unexpected:
        print(f'WARNING — unexpected extra keys in ckpt   : {unexpected}')
    print('Pretrained weights loaded successfully into CA generator.')
    return model


In [ ]:


# Build CA generator and load pretrained weights
net_g = RRDBNetCA(
    num_in_ch      = config.num_in_ch,
    num_out_ch     = config.num_out_ch,
    scale          = config.scale_factor,
    num_feat       = config.num_feat,
    num_block      = config.num_block,
    num_grow_ch    = config.num_grow_ch,
    ca_reduction   = config.ca_reduction,
    use_ca_after_trunk = config.use_ca_after_trunk,
    use_ca_after_up1   = config.use_ca_after_up1,
    use_ca_after_up2   = config.use_ca_after_up2,
 )

net_g = load_pretrained_generator_ca(net_g, config.pretrained_model_path, device='cpu')
net_g = net_g.to(device)
print(f'\nGenerator-CA moved to {device}')

# Discriminator
net_d = build_discriminator(config).to(device)
print(f'Discriminator ({config.discriminator_type}) on {device}')

# EMA copy
net_g_ema = copy.deepcopy(net_g).eval()
for p in net_g_ema.parameters():
    p.requires_grad = False

@torch.no_grad()
def update_ema(ema_model, model, decay=0.999):
    for p_ema, p in zip(ema_model.parameters(), model.parameters()):
        p_ema.data.mul_(decay).add_(p.data, alpha=1.0 - decay)

print('EMA model initialised.')


## 6. Loss Functions


In [ ]:

class VGGFeatureExtractor(nn.Module):
    def __init__(self, feature_layer=35, use_input_norm=True):
        super().__init__()
        vgg = vgg19(weights=VGG19_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(vgg.features.children())[:feature_layer])
        for p in self.features.parameters():
            p.requires_grad = False
        self.use_input_norm = use_input_norm
        if use_input_norm:
            self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
            self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        if self.use_input_norm:
            x = (x - self.mean) / self.std
        return self.features(x)


class L1PixelLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.L1Loss()
    def forward(self, sr, hr):
        return self.loss(sr, hr)


class PerceptualLoss(nn.Module):
    def __init__(self, feature_layer=35):
        super().__init__()
        self.vgg  = VGGFeatureExtractor(feature_layer=feature_layer)
        self.loss = nn.L1Loss()
    def forward(self, sr, hr):
        return self.loss(self.vgg(sr), self.vgg(hr))


class GANLoss(nn.Module):
    def __init__(self, target_real=1.0, target_fake=0.0):
        super().__init__()
        self.real_label = target_real
        self.fake_label = target_fake
        self.bce = nn.BCEWithLogitsLoss()

    def _target(self, pred, is_real):
        val = self.real_label if is_real else self.fake_label
        return torch.full_like(pred, val)

    def forward(self, d_real, d_fake, is_disc):
        real_logit = d_real - d_fake.mean()
        fake_logit = d_fake - d_real.mean()
        if is_disc:
            loss = (self.bce(real_logit, self._target(real_logit, True)) +
                    self.bce(fake_logit, self._target(fake_logit, False))) / 2
        else:
            loss = (self.bce(real_logit, self._target(real_logit, False)) +
                    self.bce(fake_logit, self._target(fake_logit, True))) / 2
        return loss


cri_pixel      = L1PixelLoss().to(device)
cri_perceptual = PerceptualLoss(feature_layer=35).to(device)
cri_gan        = GANLoss().to(device)
print('Loss functions ready.')


## 7. Evaluation Metrics (PSNR + SSIM + LPIPS)


In [ ]:

lpips_fn = lpips.LPIPS(net='alex').to(device).eval()
for p in lpips_fn.parameters():
    p.requires_grad = False
print('LPIPS model loaded.')


def calculate_psnr(sr, hr, max_val=1.0):
    mse = F.mse_loss(sr, hr)
    if mse == 0:
        return float('inf')
    return (20 * torch.log10(torch.tensor(max_val) / torch.sqrt(mse))).item()


def calculate_ssim(sr, hr):
    from skimage.metrics import structural_similarity as ssim
    sr_np = sr.squeeze().permute(1, 2, 0).cpu().numpy().clip(0, 1)
    hr_np = hr.squeeze().permute(1, 2, 0).cpu().numpy().clip(0, 1)
    return ssim(sr_np, hr_np, data_range=1.0, channel_axis=2)


@torch.no_grad()
def calculate_lpips(sr, hr):
    sr_scaled = sr * 2.0 - 1.0
    hr_scaled = hr * 2.0 - 1.0
    return lpips_fn(sr_scaled, hr_scaled).item()


## 8. Fine-Tuner with CA-aware Checkpoint Handling


In [ ]:

class RealESRGANFineTunerCA:

    def __init__(self, cfg, net_g, net_d, net_g_ema, device):
        self.cfg       = cfg
        self.device    = device
        self.net_g     = net_g
        self.net_d     = net_d
        self.net_g_ema = net_g_ema

        # Layer-wise learning rates: CA linear layers train from scratch → 10× higher LR.
        _ca_params   = [p for n, p in self.net_g.named_parameters() if ('ca_' in n or '.ca.' in n)]
        _conv_params = [p for n, p in self.net_g.named_parameters() if not ('ca_' in n or '.ca.' in n)]
        self.opt_g = optim.Adam(
            [{'params': _conv_params, 'lr': cfg.lr_g},
             {'params': _ca_params,   'lr': cfg.lr_g_ca}],
            betas=cfg.betas)
        self.opt_d = optim.Adam(self.net_d.parameters(), lr=cfg.lr_d, betas=cfg.betas)

        self.sched_g = optim.lr_scheduler.CosineAnnealingLR(
            self.opt_g, T_max=cfg.num_epochs, eta_min=cfg.lr_min)
        self.sched_d = optim.lr_scheduler.CosineAnnealingLR(
            self.opt_d, T_max=cfg.num_epochs, eta_min=cfg.lr_min)

        self.scaler_g = amp.GradScaler()
        self.scaler_d = amp.GradScaler()

        # TensorBoard — copy previous events from config.prev_log_dir if set
        self._init_tensorboard(cfg)

        self.epoch       = 0
        self.global_step = 0
        self.best_psnr   = 0.0
        self.best_lpips  = float('inf')
        self.best_percep = self.best_lpips  # backward-compatible alias

        self.discriminator_frozen = False

    # TensorBoard
    def _init_tensorboard(self, cfg):
        if cfg.prev_log_dir and os.path.isdir(cfg.prev_log_dir):
            for fname in os.listdir(cfg.prev_log_dir):
                src = os.path.join(cfg.prev_log_dir, fname)
                dst = os.path.join(cfg.log_dir, fname)
                if os.path.isfile(src) and not os.path.exists(dst):
                    shutil.copy2(src, dst)
            print(f'Copied previous TensorBoard events from {cfg.prev_log_dir}')
        self.writer = SummaryWriter(cfg.log_dir, purge_step=None)

    #Utilities
    @staticmethod
    def _grad_norm(model):
        total = 0.0
        for p in model.parameters():
            if p.grad is not None:
                total += p.grad.data.norm(2).item() ** 2
        return total ** 0.5

    @staticmethod
    def _resolve_ckpt_path(path_or_name, checkpoint_dir):
        """Accept full absolute path, existing relative path, or filename in checkpoint_dir."""
        if os.path.isabs(path_or_name) and os.path.isfile(path_or_name):
            return path_or_name
        if os.path.isfile(path_or_name):
            return path_or_name
        candidate = os.path.join(checkpoint_dir, path_or_name)
        if os.path.isfile(candidate):
            return candidate
        raise FileNotFoundError(
            f"Checkpoint not found: '{path_or_name}' (also tried '{candidate}')")

    def set_discriminator_frozen(self, freeze=True, reason=''):
        if freeze == self.discriminator_frozen:
            return
        for p in self.net_d.parameters():
            p.requires_grad = not freeze
        if freeze:
            self.net_d.eval()
        else:
            self.net_d.train()
        self.discriminator_frozen = freeze
        state = 'FROZEN' if freeze else 'UNFROZEN'
        extra = f' ({reason})' if reason else ''
        print(f'>>> Discriminator {state}{extra}')

    def _should_freeze_discriminator(self, epoch, use_gan):
        if not use_gan:
            return True
        if getattr(self.cfg, 'freeze_discriminator_always', False):
            return True

        start = getattr(self.cfg, 'freeze_discriminator_start_epoch', None)
        end   = getattr(self.cfg, 'freeze_discriminator_end_epoch', None)
        if start is None or end is None:
            return False

        epoch_1based = epoch + 1
        return start <= epoch_1based <= end

    def _gan_lambda_for_epoch(self, epoch, use_gan):
        if not use_gan:
            return 0.0
        if epoch < int(self.cfg.gan_rampup_end_epoch):
            return float(self.cfg.lambda_gan_low)
        return float(self.cfg.lambda_gan_high)

    #  Single training step
    def train_step(self, lr_imgs, hr_imgs, use_gan, freeze_d=False, gan_lambda=0.0,
                   accum_steps=1, accum_index=0, do_optimizer_step=True):
        lr_imgs = lr_imgs.to(self.device, non_blocking=True)
        hr_imgs = hr_imgs.to(self.device, non_blocking=True)

        d_loss_val = 0.0
        if use_gan:
            self.opt_d.zero_grad(set_to_none=True)
            with torch.no_grad():
                sr_imgs = self.net_g(lr_imgs)
            with amp.autocast(device_type='cuda'):
                d_real = self.net_d(hr_imgs)
                d_fake = self.net_d(sr_imgs.detach())
                d_loss = cri_gan(d_real, d_fake, is_disc=True)
            d_loss_val = d_loss.item()
            self.scaler_d.scale(d_loss).backward()
            self.scaler_d.step(self.opt_d)
            self.scaler_d.update()

        if accum_index == 0:
            self.opt_g.zero_grad(set_to_none=True)
        with amp.autocast(device_type='cuda'):
            sr_imgs  = self.net_g(lr_imgs)
            l_pixel  = cri_pixel(sr_imgs, hr_imgs)       * self.cfg.lambda_pixel
            l_percep = cri_perceptual(sr_imgs, hr_imgs)  * self.cfg.lambda_perceptual
            if use_gan:
                d_real = self.net_d(hr_imgs).detach()
                d_fake = self.net_d(sr_imgs)
                l_gan  = cri_gan(d_real, d_fake, is_disc=False) * gan_lambda
            else:
                l_gan  = torch.zeros(1, device=self.device).squeeze()
            g_loss = l_pixel + l_percep + l_gan

        self.scaler_g.scale(g_loss / accum_steps).backward()

        g_grad_norm = 0.0
        if do_optimizer_step:
            self.scaler_g.unscale_(self.opt_g)
            g_grad_norm = self._grad_norm(self.net_g)
            if g_grad_norm != g_grad_norm or g_grad_norm == float('inf'):
                g_grad_norm = 0.0
            torch.nn.utils.clip_grad_norm_(self.net_g.parameters(), max_norm=5.0)
            self.scaler_g.step(self.opt_g)
            self.scaler_g.update()
            update_ema(self.net_g_ema, self.net_g, self.cfg.ema_decay)

        return {
            'd_loss':      d_loss_val,
            'g_loss':      g_loss.item(),
            'pixel':       l_pixel.item(),
            'perceptual':  l_percep.item(),
            'gan':         l_gan.item() if use_gan else 0.0,
            'g_grad_norm': g_grad_norm,
            'gan_lambda':  gan_lambda if use_gan else 0.0,
        }, sr_imgs.detach()

    #  Validation
    @torch.no_grad()
    def validate(self, val_loader):
        self.net_g_ema.eval()
        psnr_sum, ssim_sum, lpips_sum, n = 0.0, 0.0, 0.0, 0
        for lr, hr, _ in val_loader:
            lr = lr.to(self.device, non_blocking=True)
            hr = hr.to(self.device, non_blocking=True)
            with amp.autocast(device_type='cuda'):
                sr = self.net_g_ema(lr).clamp(0, 1)
            h, w = hr.shape[2:]
            sr   = sr[:, :, :h, :w]
            psnr_sum  += calculate_psnr(sr, hr)
            ssim_sum  += calculate_ssim(sr, hr)
            lpips_sum += calculate_lpips(sr, hr)
            n += 1
        return {
            'psnr':  psnr_sum  / max(n, 1),
            'ssim':  ssim_sum  / max(n, 1),
            'lpips': lpips_sum / max(n, 1),
        }

    # Image logging
    @torch.no_grad()
    def _log_images(self, val_loader, epoch):
        self.net_g_ema.eval()
        lr, hr, _ = next(iter(val_loader))
        lr = lr.to(self.device)
        hr = hr.to(self.device)
        with amp.autocast(device_type='cuda'):
            sr = self.net_g_ema(lr).clamp(0, 1)
        h, w  = hr.shape[2:]
        sr    = sr[:, :, :h, :w]
        lr_up = F.interpolate(lr, size=(h, w), mode='bilinear', align_corners=False)
        grid  = torch.cat([lr_up, sr, hr], dim=3)
        self.writer.add_images('Images/LR_SR_HR', grid, epoch)

    #  Main training loop
    def train(self, train_loader, val_loader=None):
        cfg = self.cfg
        total_start = time.time()
        print(f'Starting CA fine-tuning for {cfg.num_epochs} epochs ...')
        print(f'GAN warmup: first {cfg.gan_start_epoch} epochs pixel+perceptual only')
        print(f'GAN ramp-up end epoch: {cfg.gan_rampup_end_epoch}')
        print(f'Gradient accumulation steps: {cfg.grad_accum_steps}')
        print(f'Image logging every {cfg.image_log_interval} epochs\n')

        for epoch in range(self.epoch, cfg.num_epochs):
            epoch_start = time.time()
            self.net_g.train()
            use_gan = epoch >= cfg.gan_start_epoch
            gan_lambda = self._gan_lambda_for_epoch(epoch, use_gan)

            freeze_d = self._should_freeze_discriminator(epoch, use_gan)
            self.set_discriminator_frozen(freeze_d, reason=f'epoch {epoch+1}')

            if epoch == cfg.gan_start_epoch and cfg.gan_start_epoch > 0:
                print(f'\n>>> Epoch {epoch+1}: Enabling GAN training <<<')

            if not use_gan:
                phase = 'Warmup'
            elif freeze_d:
                phase = 'GAN (D frozen)'
            else:
                phase = 'GAN'

            epoch_losses = {k: 0.0 for k in ('d_loss','g_loss','pixel','perceptual','gan','g_grad_norm')}
            grad_norm_steps = 0
            d_loss_steps = 0

            pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f'Epoch {epoch+1}/{cfg.num_epochs} [{phase}]')
            for batch_idx, (lr, hr, _) in pbar:
                accum_steps = max(1, int(cfg.grad_accum_steps))
                accum_index = batch_idx % accum_steps
                group_start = batch_idx - accum_index
                group_size = min(accum_steps, len(train_loader) - group_start)
                do_optimizer_step = (accum_index == group_size - 1)

                losses, _ = self.train_step(
                    lr, hr, use_gan, freeze_d=freeze_d, gan_lambda=gan_lambda,
                    accum_steps=group_size, accum_index=accum_index,
                    do_optimizer_step=do_optimizer_step)

                if do_optimizer_step:
                    grad_norm_steps += 1
                    if use_gan and not freeze_d:
                        d_loss_steps += 1
                    for k in epoch_losses:
                        epoch_losses[k] += losses[k]
                pbar.set_postfix(D=f"{losses['d_loss']:.4f}",
                                 G=f"{losses['g_loss']:.4f}",
                                 Pix=f"{losses['pixel']:.4f}",
                                 Gλ=f"{losses['gan_lambda']:.4f}")

                if do_optimizer_step:
                    self.global_step += 1
                    if self.global_step % cfg.log_interval == 0:
                        for k in ('g_loss','d_loss','pixel','perceptual','gan'):
                            self.writer.add_scalar(f'Train/{k}', losses[k], self.global_step)
                        self.writer.add_scalar('Train/gan_lambda', losses['gan_lambda'], self.global_step)
                        if use_gan:
                            self.writer.add_scalar('GAN/Generator', losses['gan'], self.global_step)
                            self.writer.add_scalar('GAN/Discriminator', losses['d_loss'], self.global_step)
                        self.writer.add_scalar('Gradients/Generator_norm', losses['g_grad_norm'], self.global_step)
                        # Log separate LR curves for conv (pretrained) and CA (new) param groups
                        self.writer.add_scalar('LR/generator_conv', self.opt_g.param_groups[0]['lr'], self.global_step)
                        self.writer.add_scalar('LR/generator_ca',   self.opt_g.param_groups[1]['lr'], self.global_step)
                        self.writer.add_scalar('LR/discriminator',  self.opt_d.param_groups[0]['lr'], self.global_step)

            for k in ('g_loss','pixel','perceptual','gan','g_grad_norm'):
                epoch_losses[k] /= max(grad_norm_steps, 1)
            if d_loss_steps > 0:
                epoch_losses['d_loss'] /= d_loss_steps
            else:
                epoch_losses['d_loss'] = float('nan')

            self.writer.add_scalar('TrainEpoch/g_loss_avg', epoch_losses['g_loss'], epoch)
            if d_loss_steps > 0:
                self.writer.add_scalar('TrainEpoch/d_loss_avg', epoch_losses['d_loss'], epoch)
            self.writer.add_scalar('TrainEpoch/pixel_avg', epoch_losses['pixel'], epoch)
            self.writer.add_scalar('TrainEpoch/perceptual_avg', epoch_losses['perceptual'], epoch)
            self.writer.add_scalar('TrainEpoch/gan_avg', epoch_losses['gan'], epoch)
            self.writer.add_scalar('TrainEpoch/g_grad_norm_avg', epoch_losses['g_grad_norm'], epoch)
            self.writer.add_scalar('TrainEpoch/gan_lambda', gan_lambda, epoch)

            epoch_time = time.time() - epoch_start
            elapsed    = time.time() - total_start

            print(f"\nEpoch {epoch+1}  D={epoch_losses['d_loss']:.4f}  "
                  f"G={epoch_losses['g_loss']:.4f}  "
                  f"Pix={epoch_losses['pixel']:.4f}  "
                  f"Perc={epoch_losses['perceptual']:.4f}  "
                  f"GAN={epoch_losses['gan']:.4f}  "
                  f"Gλ={gan_lambda:.4f}  "
                  f"GradNorm={epoch_losses['g_grad_norm']:.4f}  "
                  f"[{epoch_time:.0f}s / total {elapsed/60:.1f}min]")

            self.writer.add_scalar('Time/epoch_seconds', epoch_time, epoch)
            self.writer.add_scalar('Time/total_minutes', elapsed / 60, epoch)

            if val_loader and (epoch + 1) % cfg.image_log_interval == 0:
                self._log_images(val_loader, epoch)
                print('  Logged LR/SR/HR images to TensorBoard')

            val_metrics = None
            if val_loader and (epoch + 1) % cfg.val_interval == 0:
                val_metrics = self.validate(val_loader)
                print(f"  Val  PSNR={val_metrics['psnr']:.2f} dB  "
                      f"SSIM={val_metrics['ssim']:.4f}  "
                      f"LPIPS={val_metrics['lpips']:.4f}")
                self.writer.add_scalar('Val/PSNR',  val_metrics['psnr'],  epoch)
                self.writer.add_scalar('Val/SSIM',  val_metrics['ssim'],  epoch)
                self.writer.add_scalar('Val/LPIPS', val_metrics['lpips'], epoch)

                if val_metrics['psnr'] > self.best_psnr:
                    self.best_psnr = val_metrics['psnr']
                    self.save_checkpoint('best_psnr_model.pth', val_metrics, epoch_losses)
                    print(f"  New best PSNR: {self.best_psnr:.2f} dB")

                if use_gan and val_metrics['lpips'] < self.best_lpips:
                    self.best_lpips = val_metrics['lpips']
                    self.best_percep = self.best_lpips
                    self.save_checkpoint('best_perceptual_model.pth', val_metrics, epoch_losses)
                    print(f'  New best perceptual model (LPIPS): {self.best_lpips:.4f}')

            self.sched_g.step()
            if use_gan:
                self.sched_d.step()

            if (epoch + 1) % cfg.save_interval == 0:
                self.save_checkpoint(f'checkpoint_epoch_{epoch+1}.pth', val_metrics, epoch_losses)

            if getattr(cfg, 'tb_flush_every_epoch', False):
                self.writer.flush()

            self.epoch = epoch + 1

        total_time = time.time() - total_start
        print(f'\nFine-tuning complete!  Total time: {total_time/3600:.2f} hours')
        self.writer.close()

    #  Checkpoint I/O
    def save_checkpoint(self, filename, val_metrics=None, epoch_losses=None):
        path = os.path.join(self.cfg.checkpoint_dir, filename)
        ckpt = {
            'epoch':         self.epoch,
            'global_step':   self.global_step,
            'best_psnr':     self.best_psnr,
            'best_lpips':    self.best_lpips,
            'best_percep':   self.best_percep,
            'params':        self.net_g.state_dict(),
            'params_ema':    self.net_g_ema.state_dict(),
            'discriminator': self.net_d.state_dict(),
            'opt_g':         self.opt_g.state_dict(),
            'opt_d':         self.opt_d.state_dict(),
            'sched_g':       self.sched_g.state_dict(),
            'sched_d':       self.sched_d.state_dict(),
            'scaler_g':      self.scaler_g.state_dict(),
            'scaler_d':      self.scaler_d.state_dict(),
            'config': {
                'scale_factor':      self.cfg.scale_factor,
                'num_feat':          self.cfg.num_feat,
                'num_block':         self.cfg.num_block,
                'num_grow_ch':       self.cfg.num_grow_ch,
                'ca_reduction':      self.cfg.ca_reduction,
                'use_ca_after_trunk': self.cfg.use_ca_after_trunk,
                'use_ca_after_up1':   self.cfg.use_ca_after_up1,
                'use_ca_after_up2':   self.cfg.use_ca_after_up2,
                'gan_rampup_end_epoch': self.cfg.gan_rampup_end_epoch,
                'grad_accum_steps':  self.cfg.grad_accum_steps,
                'lr_g':              self.cfg.lr_g,
                'lr_g_ca':           self.cfg.lr_g_ca,
                'lr_d':              self.cfg.lr_d,
                'lambda_pixel':      self.cfg.lambda_pixel,
                'lambda_perceptual': self.cfg.lambda_perceptual,
                'lambda_gan_low':    self.cfg.lambda_gan_low,
                'lambda_gan_high':   self.cfg.lambda_gan_high,
                'ema_decay':         self.cfg.ema_decay,
                'freeze_discriminator_always': self.cfg.freeze_discriminator_always,
                'freeze_discriminator_start_epoch': self.cfg.freeze_discriminator_start_epoch,
                'freeze_discriminator_end_epoch': self.cfg.freeze_discriminator_end_epoch,
            },
        }
        if val_metrics:
            ckpt['val_metrics']   = val_metrics
        if epoch_losses:
            ckpt['epoch_losses']  = epoch_losses
        torch.save(ckpt, path)
        print(f'  Saved: {path}')

    def load_checkpoint(self, path_or_name):

        path = self._resolve_ckpt_path(path_or_name, self.cfg.checkpoint_dir)
        ckpt = torch.load(path, map_location=self.device, weights_only=False)

        missing_g, _ = self.net_g.load_state_dict(ckpt['params'], strict=False)
        self.net_g_ema.load_state_dict(ckpt['params_ema'], strict=False)
        self.net_d.load_state_dict(ckpt['discriminator'])
        # Guard against param-group count mismatch when resuming from a single-group checkpoint.
        try:
            self.opt_g.load_state_dict(ckpt['opt_g'])
        except (ValueError, KeyError):
            print('  opt_g state could not be restored (param group mismatch) — starting opt_g fresh')
        self.opt_d.load_state_dict(ckpt['opt_d'])
        self.sched_g.load_state_dict(ckpt['sched_g'])
        self.sched_d.load_state_dict(ckpt['sched_d'])
        self.epoch       = ckpt['epoch']
        self.global_step = ckpt['global_step']
        self.best_psnr   = ckpt.get('best_psnr', 0.0)
        self.best_lpips  = ckpt.get('best_lpips', ckpt.get('best_percep', float('inf')))
        self.best_percep = self.best_lpips
        if 'scaler_g' in ckpt:
            self.scaler_g.load_state_dict(ckpt['scaler_g'])
        if 'scaler_d' in ckpt:
            self.scaler_d.load_state_dict(ckpt['scaler_d'])

        ca_missing = [k for k in missing_g if ('ca_' in k or '.ca.' in k)]
        print(f'Resumed from {path}  (epoch {self.epoch}, step {self.global_step}, '
              f'best_psnr {self.best_psnr:.2f})')
        if ca_missing:
            print(f'  {len(ca_missing)} CA keys kept at init (checkpoint has no CA weights — expected for baseline ckpts)')


## 9. Initialize and Train


In [ ]:

# Create DataLoaders
train_loader, val_loader = create_dataloaders(config)
print(f'Train: {len(train_loader.dataset)} images  |  Val: {len(val_loader.dataset)} images')

# Create Trainer  (reads config.prev_log_dir during __init__)
trainer = RealESRGANFineTunerCA(config, net_g, net_d, net_g_ema, device)
print('Trainer ready.')


In [ ]:

from RealESRGAN.train import run_training

trainer = run_training(config)


In [ ]:

!zip -r output.zip /kaggle/working/experiments


## 10. Inference


In [ ]:

import gc


def load_finetuned_model_ca(weight_path, cfg, device):
    """Load a CA checkpoint (or a baseline checkpoint) into RRDBNetCA."""
    model = RRDBNetCA(
        num_in_ch      = cfg.num_in_ch,
        num_out_ch     = cfg.num_out_ch,
        scale          = cfg.scale_factor,
        num_feat       = cfg.num_feat,
        num_block      = cfg.num_block,
        num_grow_ch    = cfg.num_grow_ch,
        ca_reduction   = cfg.ca_reduction,
        use_ca_after_trunk = cfg.use_ca_after_trunk,
        use_ca_after_up1   = cfg.use_ca_after_up1,
        use_ca_after_up2   = cfg.use_ca_after_up2,
    )

    ckpt = torch.load(weight_path, map_location='cpu', weights_only=False)

    if 'params_ema' in ckpt:
        model.load_state_dict(ckpt['params_ema'], strict=False)
        print(f'Loaded EMA weights from {weight_path}')
    elif 'params' in ckpt:
        model.load_state_dict(ckpt['params'], strict=False)
        print(f'Loaded params from {weight_path}')
    else:
        model.load_state_dict(ckpt, strict=False)

    epoch_num   = ckpt.get('epoch')      if isinstance(ckpt, dict) else None
    val_metrics = ckpt.get('val_metrics') if isinstance(ckpt, dict) else None
    del ckpt
    gc.collect()

    if epoch_num   is not None: print(f'Checkpoint epoch: {epoch_num}')
    if val_metrics is not None:
        print(f'Val PSNR: {val_metrics["psnr"]:.2f} dB  |  SSIM: {val_metrics["ssim"]:.4f}'
              f'  |  LPIPS: {val_metrics.get("lpips", "N/A")}')

    model = model.to(device).eval()
    return model


def _run_sr(model, lr_tensor, device):
    lr_gpu = lr_tensor.to(device)
    with torch.no_grad(), amp.autocast(device_type='cuda'):
        sr = model(lr_gpu).float().clamp(0, 1).cpu()
    del lr_gpu
    return sr


def _load_run_unload_ca(weight_path, cfg, lr_tensor, device):
    model = load_finetuned_model_ca(weight_path, cfg, device)
    sr    = _run_sr(model, lr_tensor, device)
    del model
    gc.collect()
    return sr


def _free_training_models():
    for name in ['net_g', 'net_d', 'net_g_ema', 'cri_perceptual', 'cri_pixel', 'cri_gan', 'trainer']:
        obj = globals().get(name)
        if obj is not None and hasattr(obj, 'cpu'):
            try:
                obj.cpu()
            except Exception:
                pass
    gc.collect()
    print('Freed training models from GPU.')


def inference_compare_ca(cfg, lr_dir, hr_dir, device, mode='all'):
    """
    mode:
        'pretrained'  — LR / Pretrained-CA / HR
        'psnr'        — LR / Pretrained-CA / Best PSNR-CA / HR
        'all'         — LR / Pretrained-CA / Best PSNR-CA / Perceptual-CA / HR
    """
    _free_training_models()
    to_tensor = transforms.ToTensor()

    lr_images = sorted(glob.glob(os.path.join(lr_dir, '*')))
    lr_images = [p for p in lr_images
                 if os.path.splitext(p)[1].lower() in ('.png','.jpg','.jpeg','.bmp','.tif','.tiff')]
    if not lr_images:
        print("No LR images found.")
        return
    lr_path = random.choice(lr_images)
    lr_name = os.path.basename(lr_path)
    hr_name = lr_name.replace('x1', 'x4')
    hr_path = os.path.join(hr_dir, hr_name)

    lr_img    = Image.open(lr_path).convert('RGB')
    hr_img    = Image.open(hr_path).convert('RGB') if os.path.exists(hr_path) else None
    lr_tensor = to_tensor(lr_img).unsqueeze(0)

    results = {}

    print('Running pretrained model (CA)...')
    results['Pretrained-CA'] = _load_run_unload_ca(cfg.pretrained_model_path, cfg, lr_tensor, device)

    if mode in ('psnr', 'all'):
        psnr_path = os.path.join(cfg.checkpoint_dir, 'best_psnr_model.pth')
        if os.path.exists(psnr_path):
            print('Running best PSNR-CA model...')
            results['Best PSNR-CA'] = _load_run_unload_ca(psnr_path, cfg, lr_tensor, device)
        else:
            print(f'Best PSNR checkpoint not found: {psnr_path}')

    if mode == 'all':
        perc_path = os.path.join(cfg.checkpoint_dir, 'best_perceptual_model.pth')
        if os.path.exists(perc_path):
            print('Running best perceptual-CA model...')
            results['Perceptual-CA'] = _load_run_unload_ca(perc_path, cfg, lr_tensor, device)
        else:
            print(f'Best perceptual checkpoint not found: {perc_path}')

    hr_tensor   = to_tensor(hr_img).unsqueeze(0) if hr_img is not None else None
    metric_strs = []
    if hr_tensor is not None:
        h, w = hr_tensor.shape[2:]
        for name, sr_cpu in results.items():
            sr_crop   = sr_cpu[:, :, :h, :w]
            psnr_val  = calculate_psnr(sr_crop, hr_tensor)
            ssim_val  = calculate_ssim(sr_crop, hr_tensor)
            sr_gpu    = sr_crop.to(device)
            hr_gpu    = hr_tensor.to(device)
            lpips_val = calculate_lpips(sr_gpu, hr_gpu)
            del sr_gpu, hr_gpu
            metric_strs.append(
                f'{name}: PSNR={psnr_val:.2f}, SSIM={ssim_val:.4f}, LPIPS={lpips_val:.4f}')

    has_hr = hr_img is not None
    ncols  = 1 + len(results) + (1 if has_hr else 0)
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))
    if ncols == 1:
        axes = [axes]

    col = 0
    axes[col].imshow(np.array(lr_img))
    axes[col].set_title(f'LR ({lr_img.size[0]}x{lr_img.size[1]})')
    axes[col].axis('off')
    col += 1

    for name, sr_cpu in results.items():
        sr_np = sr_cpu.squeeze().permute(1, 2, 0).numpy()
        axes[col].imshow(sr_np)
        axes[col].set_title(f'{name} ({sr_np.shape[1]}x{sr_np.shape[0]})')
        axes[col].axis('off')
        col += 1

    if has_hr:
        axes[col].imshow(np.array(hr_img))
        axes[col].set_title(f'HR GT ({hr_img.size[0]}x{hr_img.size[1]})')
        axes[col].axis('off')

    if metric_strs:
        fig.suptitle(f'{lr_name}\n' + '  |  '.join(metric_strs), fontsize=11)

    plt.tight_layout()
    plt.show()

    del lr_tensor, results
    if hr_tensor is not None:
        del hr_tensor
    gc.collect()


In [ ]:

# mode='pretrained'  — LR / Pretrained-CA / HR
# mode='psnr'        — LR / Pretrained-CA / Best PSNR-CA / HR
# mode='all'         — LR / Pretrained-CA / Best PSNR-CA / Perceptual-CA / HR

inference_compare_ca(config, config.val_lr_dir, config.val_hr_dir, device, mode='all')


In [ ]:
# Modular folder inference (RealESRGAN package)
from RealESRGAN.infer import run_inference_folder

# Update these paths as needed
weight_path = "experiments/checkpoints/best_psnr_model.pth"
input_dir = config.val_lr_dir
output_dir = "imageoutputSRGenerated"

saved_paths = run_inference_folder(
    cfg=config,
    weight_path=weight_path,
    input_dir=input_dir,
    output_dir=output_dir,
    recursive=False,
    keep_structure=False,
 )

print(f"Inference complete. Saved {len(saved_paths)} images to: {output_dir}")